#### Purpose: Ingest raw graduate student longitudinal data, flag and fix quality issues, construct metrics, and export a clean parquet file for reporting and modeling.

#### Git commit history for this file:
##### Commit 1: Skeleton and raw loading of data (done)
##### Commit 2: Standardize data - cleaning
##### Commit 3: Construct financial support and career metrics
##### Commit 4: add data-quality report export

In [1]:
# importing packages
import pandas as pd
import numpy as np
from pathlib import Path
import json

### PATHS to relative project root

In [2]:
# This notebook lives in /src, so go up one level to reach the project root.
PROJECT_ROOT = Path().resolve().parent

RAW_PATH   = PROJECT_ROOT / "data" / "raw" / "grad_students_raw.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "grad_students_clean.parquet"
QA_PATH    = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"

print("Project root resolved to:", PROJECT_ROOT)
print("Raw data path:", RAW_PATH)


Project root resolved to: /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis
Raw data path: /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/raw/grad_students_raw.csv


### Step 1: Loading data

In [3]:
## using Parquet for efficiency and reduction of column type errors
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)  # reads everything as string first, decide types later
    print(f"[LOAD] {len(df):,} x {len(df.columns)} columns loaded from {path}")
    return df

### Step 2: Standardizing data

In [4]:
BOOL_COLUMNS = ["pell_eligible", "TA_status", "GSR_status", "first_gen", "international"]

def _to_bool(series: pd.Series) -> pd.Series:
    """Convert free-text yes/no column to nullable boolean (pd.BooleanDtype)."""
    mapping = {"yes": True,
               "no": False,
               "1": True,
               "0": False,
               "true": True,
               "false": False}
    cleaned = series.str.strip().str.lower().map(mapping)
    return cleaned.astype(pd.BooleanDtype())

def standardize_booleans(df: pd.DataFrame) -> pd.DataFrame:
    """Apply bool normalization to all flag columns."""
    for col in BOOL_COLUMNS:
        df[col] = _to_bool(df[col])
    print(f"[STANDARDIZE] Boolean columns normalized")
    return df

def standardize_numerics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast numeric columns; coerce unparseable strings to NaN so we know
    exactly where the gaps are rather than silently dropping rows.
    """
    numeric_cols = {
        "grant_amt":          float,
        "gpa":                float,
        "units_enrolled":     int,
        "stipend_amt":        float,
        "time_to_degree":     float,
        "salary_offer":       float,
        "months_to_employment": float,
        "year_in_program":    int,
        "term_year":          int,
    }
    for col, dtype in numeric_cols.items():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if dtype == int:
            df[col] = df[col].astype("Int64")
        else:
            df[col] = df[col].astype(float)
    print(f"[STANDARDIZE] Numeric columns cast")
    return df

def standardize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Title-case free-text columns; strip whitespace."""
    str_cols = ["program", "thesis_status", "employment_outcome",
                "employer_type", "job_title", "industry", "gender",
                "ethnicity", "academic_term"]
    for col in str_cols:
        df[col] = df[col].str.strip().str.title()
    df["last_name"]  = df["last_name"].str.strip().str.upper()
    df["first_name"] = df["first_name"].str.strip().str.title()
    print(f"[STANDARDIZE] String columns cleaned")
    return df


### Step 3: Creating metrics
#### new columns created
- term_label: human-readable "Fall 2019"
- any_support: TA or GSR in that term
- financial_aid_flag: pell_eligible OR grant_amt > 0
- total_financial_support: grant_amt + stipend_amt
- gpa_risk_flag: GPA below 3.0 (probation threshold)
- thesis_stage_num: ordinal thesis progress
- degree_program_type: PhD or MS
- cohort_year: first term_year a student appears
- year_since_cohort: term_year minus cohort_year
- pell_x_first_gen: interaction of both pell-eligible AND first generation status
- log_salary: log-transformed salary for modeling
- employed_flag: binary outcome for classification modeling

In [5]:
df = load_raw(RAW_PATH)
df = standardize_booleans(df)
df = standardize_numerics(df)
df = standardize_strings(df)

[LOAD] 550 x 28 columns loaded from /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/raw/grad_students_raw.csv
[STANDARDIZE] Boolean columns normalized
[STANDARDIZE] Numeric columns cast
[STANDARDIZE] String columns cleaned


In [6]:
# Term Labels
df["term_label"] = df["academic_term"].str.strip() + " " + df["term_year"].astype(str)

In [7]:
# Support flags
df["any_support"] = (df["TA_status"].fillna(False) |
                        df["GSR_status"].fillna(False))

df["financial_aid_flag"] = (
     df["pell_eligible"].fillna(False) |
     (df["grant_amt"].fillna(0) > 0)
    )

df["total_financial_support"] = (
    df["grant_amt"].fillna(0) + df["stipend_amt"].fillna(0)
)


In [8]:
# Academic risk
df["gpa_risk_flag"] = df["gpa"] < 3.0

In [9]:
# Thesis Stage (ordinal
stage_order = {
    "Proposal": 1, "Approved": 2,
    "Writing": 3, "Defended": 4,
    "N/A": 0
}
df["thesis_stage_num"] = (
    df["thesis_status"].str.title().map(stage_order).fillna(0).astype(int)
)

In [10]:
# Degree Type
df["degree_program_type"] = np.where(
    df["program"].str.upper().str.startswith("PHD"),
    "PHD",
    "Masters"
)

In [11]:
# Cohort Year
df["cohort_year"] = df.groupby("student_id")["term_year"].transform("min")
df["years_since_cohort"] = df["term_year"].astype(int) - df["cohort_year"]

In [12]:
# Interaction pell and first gen
df["pell_x_first_gen"] = (
     df["pell_eligible"].fillna(False) & df["first_gen"].fillna(False)
).astype(int)

In [13]:
# Career outcomes
df["employed_flag"] = (
    df["employment_outcome"].str.upper() == "EMPLOYED"
    ).astype("Int64")

df["log_salary"] = np.log(df["salary_offer"].replace(0, np.nan))

print(f"[ENGINEER] {len([c for c in df.columns if c not in load_raw.__code__.co_varnames])} new features created")

[ENGINEER] 40 new features created
